In [1]:
import requests
import pandas as pd
from datetime import datetime
from google.transit import gtfs_realtime_pb2


URL = "https://realtime.gtfs.de/realtime-free.pb"

response = requests.get(URL, timeout=30)


In [2]:
def parse_gtfs_feed(response):
    """
    Parse a GTFS-RT response into a FeedMessage.

    Parameters
    ----------
    response : requests.Response
        HTTP response containing the serialized GTFS-RT feed.

    Returns
    -------
    FeedMessage
        Parsed GTFS-RT feed.
    """
    feed = gtfs_realtime_pb2.FeedMessage()
    feed.ParseFromString(response.content)

    print(f"Number of entities: {len(feed.entity)}")

    return feed

feed = parse_gtfs_feed(response)


Number of entities: 20454


In [3]:
for entity in feed.entity[:10]:
    print(entity)

id: "548398tu"
trip_update {
  trip {
    trip_id: "548398"
    start_date: "20260905"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 4
    arrival {
      delay: 467
      time: 1788560207
    }
    departure {
      delay: 467
      time: 1788560207
    }
    stop_id: "250923"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 5
    arrival {
      delay: 407
      time: 1788560207
    }
    departure {
      delay: 407
      time: 1788560207
    }
    stop_id: "72703"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 6
    arrival {
      delay: 293
      time: 1788560213
    }
    departure {
      delay: 293
      time: 1788560213
    }
    stop_id: "23884"
    schedule_relationship: SCHEDULED
  }
  stop_time_update {
    stop_sequence: 7
    arrival {
      delay: 263
      time: 1788560303
    }
    departure {
      delay: 279
      time: 1788560319
    }
    stop_id: "233325"
    sch

In [4]:
stops_df = pd.read_csv("../data/mvv_stops.csv", delimiter=";")


In [5]:
def create_stop_name_mapping(stops_df):
    """
    Create a mapping from MVV stop IDs to stop names.

    Parameters
    ----------
    stops_df : pandas.DataFrame
        DataFrame containing the MVV stop data. It must contain
        the columns "HstNummer" and "Name ohne Ort".

    Returns
    -------
    dict
        Dictionary mapping stop IDs to stop names.
    """
    stops_df["HstNummer"] = stops_df["HstNummer"].astype(str)

    stop_names = (
        stops_df
        .set_index("HstNummer")["Name ohne Ort"]
        .to_dict()
    )

    return stop_names


In [6]:
def parse_trip_updates(feed, stop_names, trip_lines):
    """
    Parse GTFS-RT trip updates into a pandas DataFrame.

    Parameters
    ----------
    feed : FeedMessage
        Parsed GTFS-RT feed containing trip updates.
    stop_names : dict
        Mapping from stop IDs to stop names.
    trip_lines : dict
        Mapping from trip IDs to line names.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing trip, line, stop, arrival,
        departure, and delay information.
    """
    rows = []

    for entity in feed.entity:
        if not entity.HasField("trip_update"):
            continue

        trip = entity.trip_update.trip

        # Get line for this trip
        line = trip_lines.get(str(trip.trip_id))

        # Ignore trips that are not part of the selected agencies
        if line is None:
            continue

        for stop in entity.trip_update.stop_time_update:

            row = {
                "trip_id": trip.trip_id,
                "start_date": trip.start_date,
                "line": line,
                "stop_id": str(stop.stop_id),
                "stop_name": stop_names.get(str(stop.stop_id)),
                "stop_sequence": stop.stop_sequence,
            }

            if stop.HasField("departure"):
                row["departure_time"] = datetime.fromtimestamp(
                    stop.departure.time
                )
                row["departure_delay"] = stop.departure.delay

            if stop.HasField("arrival"):
                row["arrival_time"] = datetime.fromtimestamp(
                    stop.arrival.time
                )
                row["arrival_delay"] = stop.arrival.delay

            rows.append(row)

    return pd.DataFrame(rows)

In [7]:
def preprocess_gtfs(data_dir, munich_agencies):
    """
    Preprocess GTFS static data for selected agencies.

    Returns
    -------
    trip_lines : dict
        Mapping from trip_id to line name.

    stop_names : dict
        Mapping from stop_id to stop name.
    """

    routes_df = pd.read_csv(f"{data_dir}/routes.txt")
    trips_df = pd.read_csv(f"{data_dir}/trips.txt")
    stops_df = pd.read_csv(f"{data_dir}/stops.txt")

    routes_df["route_id"] = routes_df["route_id"].astype(str)
    routes_df["agency_id"] = routes_df["agency_id"].astype(str)

    trips_df["trip_id"] = trips_df["trip_id"].astype(str)
    trips_df["route_id"] = trips_df["route_id"].astype(str)

    stops_df["stop_id"] = stops_df["stop_id"].astype(str)

    munich_routes = routes_df[
        routes_df["agency_id"].isin(munich_agencies)
    ]

    route_lines = (
        munich_routes
        .set_index("route_id")["route_short_name"]
        .to_dict()
    )

    munich_trips = trips_df[
        trips_df["route_id"].isin(route_lines)
    ]

    trip_lines = (
        munich_trips
        .set_index("trip_id")["route_id"]
        .map(route_lines)
        .to_dict()
    )

    stop_names = (
        stops_df
        .set_index("stop_id")["stop_name"]
        .to_dict()
    )

    return trip_lines, stop_names


In [8]:
munich_agencies = ["100", "191", "364"]

trip_lines, stop_names = preprocess_gtfs(
    "../data",
    munich_agencies
)

In [9]:
df = parse_trip_updates(
    feed,
    stop_names,
    trip_lines
)

df.head(100)

,trip_id,start_date,line,stop_id,stop_name,stop_sequence,departure_time,departure_delay,arrival_time,arrival_delay
0,1192330,20260904,190,614970,August-Everding-Straße,1,2026-09-05 00:37:33,3.0,2026-09-05 00:37:33,3.0
1,1192330,20260904,190,576765,Sankt Pius,2,2026-09-05 00:37:48,-12.0,2026-09-05 00:37:48,-12.0
2,1192330,20260904,190,686743,Grafinger Straße,3,2026-09-05 00:39:22,-8.0,2026-09-05 00:38:52,-38.0
3,1192330,20260904,190,634241,Altöttinger Straße,4,2026-09-05 00:40:34,4.0,2026-09-05 00:40:11,-19.0
4,1192330,20260904,190,561261,Schlüsselbergstraße,5,2026-09-05 00:41:42,-18.0,2026-09-05 00:41:11,-49.0
...,...,...,...,...,...,...,...,...,...,...
95,1288664,20260904,193,238930,"Haar, Hans-Stießberger-Straße",11,2026-09-05 00:59:48,-42.0,2026-09-05 00:59:22,-68.0
96,1288664,20260904,193,411746,"Haar, Ludwig-Moser-Straße",12,2026-09-05 01:00:11,-79.0,2026-09-05 01:00:11,-79.0
97,1288664,20260904,193,509917,"Haar, Jagdfeldzentrum",14,NaT,NaN,2026-09-05 01:03:30,0.0
98,566070,20260904,62,509290,Ostbahnhof,0,2026-09-05 00:38:52,-8.0,NaT,NaN


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5502 entries, 0 to 5501
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   trip_id          5502 non-null   object        
 1   start_date       5502 non-null   object        
 2   line             5502 non-null   object        
 3   stop_id          5502 non-null   object        
 4   stop_name        5502 non-null   object        
 5   stop_sequence    5502 non-null   int64         
 6   departure_time   4928 non-null   datetime64[ns]
 7   departure_delay  4928 non-null   float64       
 8   arrival_time     4759 non-null   datetime64[ns]
 9   arrival_delay    4759 non-null   float64       
dtypes: datetime64[ns](2), float64(2), int64(1), object(5)
memory usage: 430.0+ KB


In [12]:
df.to_parquet("../data/mvv_realtime.parquet", index=False)
